In [ ]:
# ============================================================
# NCF (Neural Collaborative Filtering) — Deep Learning Recommendation
# Nâng cấp từ ALS (matrix factorization) sang Neural Network
# Chạy trong notebook, dùng SparkSession `spark` đã có sẵn để đọc data từ MinIO,
# rồi chuyển sang pandas/PyTorch để train (vì PyTorch không chạy native trong Spark).
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from pyspark.sql.functions import col, count
from pyspark.sql.utils import AnalysisException

SILVER_PATH = "s3a://silver-zone/datalake/silver/eventsim"


# ──────────────────────────────────────────────
# 1. KIẾN TRÚC MODEL — NCF
# ──────────────────────────────────────────────
class NCF(nn.Module):
    """
    Neural Collaborative Filtering.
    Ý tưởng: thay vì ALS chỉ học 2 ma trận user/item rồi dot-product (tuyến tính),
    NCF học embedding rồi cho qua Multi-Layer Perceptron (MLP) để bắt được
    tương tác PHI TUYẾN giữa user và item — đây là điểm khác biệt cốt lõi so với ALS.
    """
    def __init__(self, num_users, num_items, embedding_dim=16, hidden_dims=[64, 32, 16]):
        super(NCF, self).__init__()

        # Embedding layer — tương đương với "latent factors" trong ALS,
        # nhưng ở đây sẽ được học cùng với MLP phía sau, không tách rời.
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)

        # MLP — đây là phần "Deep" của NCF, ALS không có phần này
        layers = []
        input_dim = embedding_dim * 2  # concat user_emb + item_emb
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(input_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.2))  # tránh overfit, quan trọng vì data nhỏ
            input_dim = hidden_dim
        layers.append(nn.Linear(input_dim, 1))
        layers.append(nn.Sigmoid())  # output xác suất "thích" trong khoảng [0,1]

        self.mlp = nn.Sequential(*layers)

        self._init_weights()

    def _init_weights(self):
        """Khởi tạo trọng số theo Xavier — giúp training ổn định hơn so với random mặc định."""
        nn.init.normal_(self.user_embedding.weight, std=0.01)
        nn.init.normal_(self.item_embedding.weight, std=0.01)
        for layer in self.mlp:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)

    def forward(self, user_idx, item_idx):
        u = self.user_embedding(user_idx)
        i = self.item_embedding(item_idx)
        x = torch.cat([u, i], dim=-1)
        return self.mlp(x).squeeze()


# ──────────────────────────────────────────────
# 2. DATASET — chuẩn bị data cho PyTorch DataLoader
# ──────────────────────────────────────────────
class InteractionDataset(Dataset):
    """
    Implicit feedback dataset: mỗi dòng là (user, item, label).
    label=1 nếu user đã nghe bài đó (positive), label=0 nếu chưa (negative sample).
    Đây là pattern chuẩn cho implicit feedback recsys — khác hẳn ALS chỉ cần play_count.
    """
    def __init__(self, user_indices, item_indices, labels):
        self.user_indices = torch.LongTensor(user_indices)
        self.item_indices = torch.LongTensor(item_indices)
        self.labels = torch.FloatTensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.user_indices[idx], self.item_indices[idx], self.labels[idx]


def negative_sampling(interactions_df, num_users, num_items, num_negatives=4, seed=42):
    """
    Tạo negative samples — bước QUAN TRỌNG mà ALS không cần làm thủ công (Spark ALS tự
    xử lý implicit feedback nội bộ), nhưng NCF cần negative sample rõ ràng để học
    phân biệt "user sẽ thích" vs "user sẽ không thích".

    Với mỗi positive interaction (user đã nghe bài X), sinh thêm num_negatives cặp
    (user, bài Y ngẫu nhiên mà user CHƯA nghe) làm label=0.
    """
    rng = np.random.RandomState(seed)
    positive_set = set(zip(interactions_df["user_idx"], interactions_df["item_idx"]))

    users, items, labels = [], [], []
    for u, i in zip(interactions_df["user_idx"], interactions_df["item_idx"]):
        users.append(u)
        items.append(i)
        labels.append(1)

        neg_count = 0
        attempts = 0
        while neg_count < num_negatives and attempts < num_negatives * 10:
            neg_item = rng.randint(0, num_items)
            attempts += 1
            if (u, neg_item) not in positive_set:
                users.append(u)
                items.append(neg_item)
                labels.append(0)
                neg_count += 1

    return np.array(users), np.array(items), np.array(labels)


# ──────────────────────────────────────────────
# 3. LOAD DATA TỪ MINIO (qua Spark) RỒI CHUYỂN SANG PANDAS
# ──────────────────────────────────────────────
def load_interactions(spark):
    """Đọc Silver, tạo bảng user-item interaction (giống ALS), rồi chuyển về pandas."""
    print("📦 Đọc Silver layer...")
    try:
        silver_df = spark.read.parquet(SILVER_PATH)
    except AnalysisException as e:
        print(f"⚠️ Silver chưa có dữ liệu: {e}")
        return None

    rating_df = (
        silver_df
        .filter(col("userId").isNotNull() & col("song").isNotNull())
        .groupBy("userId", "song")
        .agg(count("*").alias("play_count"))
    )

    print("   Chuyển sang pandas (data nhỏ, an toàn để collect)...")
    pdf = rating_df.toPandas()
    print(f"   Tổng interactions: {len(pdf):,} | Users: {pdf['userId'].nunique()} | Songs: {pdf['song'].nunique()}")
    return pdf


def prepare_indices(pdf):
    """Map userId/song (string) sang index số nguyên liên tục — PyTorch Embedding cần index, không nhận string."""
    user_ids = pdf["userId"].unique()
    song_ids = pdf["song"].unique()

    user_to_idx = {u: idx for idx, u in enumerate(user_ids)}
    song_to_idx = {s: idx for idx, s in enumerate(song_ids)}

    pdf = pdf.copy()
    pdf["user_idx"] = pdf["userId"].map(user_to_idx)
    pdf["item_idx"] = pdf["song"].map(song_to_idx)

    return pdf, user_to_idx, song_to_idx, len(user_ids), len(song_ids)


# ──────────────────────────────────────────────
# 4. TRAINING LOOP
# ──────────────────────────────────────────────
def train_ncf(spark, num_epochs=20, embedding_dim=16, batch_size=64, lr=0.001):
    pdf = load_interactions(spark)
    if pdf is None or len(pdf) == 0:
        print("Không có data để train.")
        return None, None

    pdf, user_to_idx, song_to_idx, num_users, num_items = prepare_indices(pdf)
    print(f"\n⚙️  Sinh negative samples (4 negative / 1 positive)...")
    users, items, labels = negative_sampling(pdf, num_users, num_items, num_negatives=4)
    print(f"   Tổng samples sau negative sampling: {len(labels):,} "
          f"(positive: {(labels==1).sum():,}, negative: {(labels==0).sum():,})")

    # Train/test split thủ công (80/20), giữ nguyên thứ tự ngẫu nhiên đã shuffle sẵn ở negative_sampling
    n = len(labels)
    rng = np.random.RandomState(42)
    perm = rng.permutation(n)
    split = int(n * 0.8)
    train_idx, test_idx = perm[:split], perm[split:]

    train_dataset = InteractionDataset(users[train_idx], items[train_idx], labels[train_idx])
    test_dataset = InteractionDataset(users[test_idx], items[test_idx], labels[test_idx])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n🖥️  Device: {device}")

    model = NCF(num_users, num_items, embedding_dim=embedding_dim).to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)

    print(f"\n🚀 Bắt đầu training {num_epochs} epochs...")
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        for u, i, y in train_loader:
            u, i, y = u.to(device), i.to(device), y.to(device)
            optimizer.zero_grad()
            pred = model(u, i)
            loss = criterion(pred, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        if (epoch + 1) % 5 == 0 or epoch == 0:
            avg_loss = total_loss / len(train_loader)
            print(f"   Epoch {epoch+1:>3}/{num_epochs} | Train Loss: {avg_loss:.4f}")

    # ── Đánh giá trên test set ──
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for u, i, y in test_loader:
            u, i = u.to(device), i.to(device)
            pred = model(u, i)
            all_preds.extend(pred.cpu().numpy())
            all_labels.extend(y.numpy())

    from sklearn.metrics import roc_auc_score, accuracy_score
    all_preds_binary = [1 if p > 0.5 else 0 for p in all_preds]
    auc = roc_auc_score(all_labels, all_preds)
    acc = accuracy_score(all_labels, all_preds_binary)

    print(f"\n{'='*50}")
    print(f"  KẾT QUẢ NCF (Test set)")
    print(f"{'='*50}")
    print(f"  AUC      : {auc:.4f}")
    print(f"  Accuracy : {acc:.4f}")
    print(f"\n  So sánh: ALS hiện tại dùng RMSE (regression), NCF dùng AUC (classification) —")
    print(f"  2 metric khác nhau, KHÔNG so sánh trực tiếp số được. Để so sánh công bằng,")
    print(f"  cần đánh giá cả 2 model bằng cùng 1 metric ranking như Recall@K hoặc NDCG@K.")

    return model, {"user_to_idx": user_to_idx, "song_to_idx": song_to_idx,
                    "num_users": num_users, "num_items": num_items}


# Cách dùng trong notebook:
# model, mappings = train_ncf(spark, num_epochs=20)